# Qwen anchor long retention compare (Colab)

Этот ноутбук запускает честный long-generation compare:
- base `Qwen/Qwen2.5-1.5B`
- anchor-biased `Qwen/Qwen2.5-1.5B`

Базовый сценарий: vegan retention prompt на длинной генерации.


> Рекомендуемый runtime в Colab: **GPU**. На CPU 500 токенов может идти слишком долго.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest


In [ ]:
%cd /content
import os
if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda'
MAX_NEW_TOKENS = 200  # first smoke run; increase to 500 after this works
MAX_LENGTH = 1024
SEED = 7
CONFLICT_THRESHOLD = 0.55
BIAS_SCALE = 1.50
PROMPT = 'You are a vegan chef. Write a detailed weekly meal plan with recipes for each day.'
OUTPUT_JSON = 'archive/colab_qwen_vegan_long_compare.json'
OUTPUT_MD = 'docs/research/colab_qwen_vegan_long_compare.md'


In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    'scripts/run_qwen_long_retention_compare.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--prompt', PROMPT,
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--max_length', str(MAX_LENGTH),
    '--seed', str(SEED),
    '--conflict_threshold', str(CONFLICT_THRESHOLD),
    '--bias_scale', str(BIAS_SCALE),
    '--output_json', OUTPUT_JSON,
    '--output_md', OUTPUT_MD,
]
print('RUNNING:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'compare script failed with code {result.returncode}')
assert Path(OUTPUT_JSON).exists(), f'missing output json: {OUTPUT_JSON}'
assert Path(OUTPUT_MD).exists(), f'missing output md: {OUTPUT_MD}'


In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
print('base lexical score:', payload['base_analysis']['lexical_score'])
print('anchor lexical score:', payload['anchor_analysis']['lexical_score'])
print('base positive hits:', payload['base_analysis']['positive_hits'])
print('anchor positive hits:', payload['anchor_analysis']['positive_hits'])
print('base negative hits:', payload['base_analysis']['negative_hits'])
print('anchor negative hits:', payload['anchor_analysis']['negative_hits'])
print('anchor bias active steps:', sum(1 for step in payload['anchor']['steps'] if step.get('bias_nonzero_anchors', 0) > 0))


In [ ]:
print('=== BASE CONTINUATION ===')
print(payload['base']['continuation_text'][:4000])
print()
print('=== ANCHOR CONTINUATION ===')
print(payload['anchor']['continuation_text'][:4000])


Если smoke run на `200` токенов прошёл, просто поменяй `MAX_NEW_TOKENS = 500` и перезапусти ячейки 5-7.
